# B05c: MLP on Tabular Data with PyTorch — COMPSCI 714

**Course Alignment:** COMPSCI 714 — AI Architecture and Design
**Related Notebooks:** [B05 - Neural Network Fundamentals](./B05%20-%20Neural%20Network%20Fundamentals.ipynb) | [B05b - Training and Optimization](./B05b%20-%20Training%20and%20Optimization%20(COMPSCI%20714).ipynb)

---

## ⚠️ Important Disclaimer

This notebook is an **independent educational resource** to help students understand PyTorch-based MLP workflows on tabular data. This is **NOT official University of Auckland course material**.

**Use Responsibly:** Understand concepts first, then implement independently for assignments. Follow the University of Auckland's Academic Integrity Policy.

---

## Learning Objectives

1. **Tabular Data Preprocessing** — ColumnTransformer, imputation, scaling, one-hot encoding
2. **Stratified vs Random Splitting** — why class balance matters
3. **PyTorch MLP for Binary Classification** — BCEWithLogitsLoss, DataLoader
4. **Training Loop Patterns** — train/val loss, accuracy tracking
5. **Evaluation Metrics** — accuracy, precision, recall, F1, confusion matrix
6. **Decision Threshold Analysis** — precision-recall trade-off
7. **Optuna Hyperparameter Tuning** — TPESampler, configurable MLP, param importance

## Prerequisites
- B05 - Neural Network Fundamentals
- B05b - Training and Optimization
- B06 - Data Preprocessing and Feature Engineering
- B07 - Model Evaluation and Performance Metrics

---

## 1. Setup — Imports and Seeds

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve
)

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

---

## 2. Load and Inspect the Dataset

We use the **German Credit Risk** dataset from OpenML — a classic binary classification task.

**Goal:** Predict whether a loan applicant is a *good* or *bad* credit risk.

### Key Concepts
- `fetch_openml` downloads the dataset directly from OpenML
- `data.data` → feature matrix X
- `data.target` → label vector y
- Mixed-type data: numeric + categorical columns
- Missing values in some categorical columns

In [ ]:
# Load German Credit Risk dataset from OpenML
data = fetch_openml(name='German-Credit-Risk-with-Target', version=1, as_frame=True)

X = data.data    # feature DataFrame
Y = data.target  # target Series

# Identify column types
numeric_features    = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print('Shape of X:', X.shape)
print('\nClass distribution:')
print(Y.value_counts())
print(f'\nNumeric features  ({len(numeric_features)}):', numeric_features)
print(f'Categorical features ({len(categorical_features)}):', categorical_features)
print('\nMissing values per column:')
print(X.isnull().sum())
print('\nFirst 5 rows:')
display(X.head())

---

## 3. Random vs Stratified Splitting

### Why Stratify?

When classes are **imbalanced** (e.g. 70% good / 30% bad), a random split may accidentally
put more bad-risk samples in one set than another. Stratified splitting **preserves the
original class ratio** in every split — making evaluation more reliable.

### Split Strategy
- 70% train / 15% validation / 15% test
- Two-step split: first cut off 30% temp, then split temp 50/50

```
Full dataset (1000)
    ├── Train  70% (700)
    ├── Valid  15% (150)   ← split from the 30% temp
    └── Test   15% (150)   ← split from the 30% temp
```

In [ ]:
# ── RANDOM SPLIT ──────────────────────────────────────────────────────────
X_train_rand, X_temp_rand, y_train_rand, y_temp_rand = train_test_split(
    X, Y, test_size=0.30, random_state=seed
)
X_valid_rand, X_test_rand, y_valid_rand, y_test_rand = train_test_split(
    X_temp_rand, y_temp_rand, test_size=0.50, random_state=seed
)

# ── STRATIFIED SPLIT ───────────────────────────────────────────────────────
X_train_strat, X_temp_strat, y_train_strat, y_temp_strat = train_test_split(
    X, Y, test_size=0.30, random_state=seed, stratify=Y
)
X_valid_strat, X_test_strat, y_valid_strat, y_test_strat = train_test_split(
    X_temp_strat, y_temp_strat, test_size=0.50, random_state=seed, stratify=y_temp_strat
)

def class_proportions(y_split):
    return y_split.value_counts(normalize=True).round(3)

for strategy, sets in [
    ('RANDOM',     (y_train_rand,  y_valid_rand,  y_test_rand)),
    ('STRATIFIED', (y_train_strat, y_valid_strat, y_test_strat)),
]:
    print(f'\n{strategy} SPLIT')
    print('=' * 50)
    for name, y_split in zip(['Train', 'Validation', 'Test'], sets):
        print(f'  {name}: {len(y_split)} samples')
        print(class_proportions(y_split).to_string(header=False))

---

## 4. Preprocessing Pipeline

### Key Rule: Fit on Train Only

Always **fit** the preprocessor on the training set, then **transform** validation and test.
Fitting on all data would cause **data leakage** — the model would indirectly see test statistics.

### Pipeline Components

| Column type | Step 1 | Step 2 |
|-------------|--------|--------|
| Numeric | `SimpleImputer(strategy='median')` | `StandardScaler()` |
| Categorical | `SimpleImputer(strategy='most_frequent')` | `OneHotEncoder(handle_unknown='ignore')` |

**Why median imputation?** Robust to outliers (e.g. Credit amount).  
**Why one-hot encoding?** Converts categories to binary columns — no ordinal assumption.

In [ ]:
# ── NUMERIC PIPELINE ──────────────────────────────────────────────────────
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# ── CATEGORICAL PIPELINE ───────────────────────────────────────────────────
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ── COLUMN TRANSFORMER ─────────────────────────────────────────────────────
preprocess_strat = ColumnTransformer(transformers=[
    ('num', numeric_pipeline,    numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

# Fit on TRAIN only, then transform all sets
X_train_proc = preprocess_strat.fit_transform(X_train_strat)
X_valid_proc  = preprocess_strat.transform(X_valid_strat)
X_test_proc   = preprocess_strat.transform(X_test_strat)

print('Feature names after preprocessing:')
print(preprocess_strat.get_feature_names_out())
print(f'\nShapes — Train: {X_train_proc.shape}, Valid: {X_valid_proc.shape}, Test: {X_test_proc.shape}')
print(f'\nOriginal features: {X_train_strat.shape[1]}  →  After OHE: {X_train_proc.shape[1]}')

---

## 5. Convert to PyTorch Tensors and DataLoaders

### Label Encoding
Binary classification needs numeric labels: `bad → 0`, `good → 1` (alphabetical order).

### Why BCEWithLogitsLoss?
- The model outputs a **raw logit** (unbounded real number)
- `BCEWithLogitsLoss` applies sigmoid internally — numerically more stable than
  applying sigmoid first then `BCELoss`
- Labels must be `float32` to match the loss function's expectation

### DataLoader
- Wraps a `TensorDataset` for mini-batch iteration
- `shuffle=True` on train set prevents the model from learning batch order

In [ ]:
# ── LABEL MAPPING ─────────────────────────────────────────────────────────
label_values = sorted(Y.unique())          # ['bad', 'good']
label_map    = {lbl: i for i, lbl in enumerate(label_values)}  # bad→0, good→1
print('Label mapping:', label_map)

y_train_num = y_train_strat.map(label_map).values
y_valid_num = y_valid_strat.map(label_map).values
y_test_num  = y_test_strat.map(label_map).values

# ── TENSORS ────────────────────────────────────────────────────────────────
X_train_t = torch.tensor(X_train_proc, dtype=torch.float32)
X_valid_t = torch.tensor(X_valid_proc,  dtype=torch.float32)
X_test_t  = torch.tensor(X_test_proc,   dtype=torch.float32)

y_train_t = torch.tensor(y_train_num, dtype=torch.float32)
y_valid_t = torch.tensor(y_valid_num, dtype=torch.float32)
y_test_t  = torch.tensor(y_test_num,  dtype=torch.float32)

# ── DATALOADERS ────────────────────────────────────────────────────────────
batch_size = 64

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(TensorDataset(X_valid_t, y_valid_t), batch_size=batch_size)
test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=batch_size)

print(f'\nBatches — Train: {len(train_loader)}, Valid: {len(valid_loader)}, Test: {len(test_loader)}')
print(f'Each training batch: {batch_size} samples (last batch may be smaller)')

---

## 6. Define a Simple MLP

### Architecture

```
Input (24 features)
    ↓
Linear(24 → 64) + ReLU
    ↓
Linear(64 → 32) + ReLU
    ↓
Linear(32 → 1)   ← single logit for binary classification
```

### Why 1 output neuron?
For binary classification with `BCEWithLogitsLoss`:
- Output is a **single logit** (not a probability yet)
- `sigmoid(logit) > 0.5` → predict class 1 (good)
- `sigmoid(logit) ≤ 0.5` → predict class 0 (bad)

Compare with multi-class: you'd use `n_classes` output neurons + `CrossEntropyLoss`.

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),   # single logit — used with BCEWithLogitsLoss
        )

    def forward(self, x):
        return self.net(x)

n_features = X_train_t.shape[1]
model = SimpleMLP(input_dim=n_features).to(device)
print(model)

# Quick sanity check: forward pass on one sample
sample = X_train_t[0].unsqueeze(0).to(device)
with torch.no_grad():
    logit = model(sample)
print(f'\nLogit for sample[0]: {logit.item():.4f}')
print(f'Probability (sigmoid): {torch.sigmoid(logit).item():.4f}')

---

## 7. Training & Evaluation Helper Functions

These reusable helpers keep the training loop clean.

| Function | What it does |
|----------|-------------|
| `train_one_epoch` | One forward + backward pass over all batches |
| `evaluate_loss` | Compute average loss on a loader (no grad) |
| `predict_binary` | Return probabilities, predictions, and true labels |
| `evaluate_binary_metrics` | Compute accuracy, precision, recall, F1, confusion matrix |

### BCEWithLogitsLoss explained
```python
loss = BCEWithLogitsLoss()(logit, label)
# Equivalent to: BCE(sigmoid(logit), label)
# But numerically stable because it avoids computing sigmoid explicitly
```

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn):
    """Run one training epoch. Returns average loss over all batches."""
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch).squeeze(1)   # shape: (batch,)
        loss   = loss_fn(logits, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss / len(loader)


@torch.no_grad()
def evaluate_loss(model, loader, loss_fn):
    """Compute average loss on a DataLoader without updating weights."""
    model.eval()
    running_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        logits = model(X_batch).squeeze(1)
        running_loss += loss_fn(logits, y_batch).item()
    return running_loss / len(loader)


@torch.no_grad()
def predict_binary(model, loader, threshold=0.5):
    """Return (probs, preds, targets) arrays for a DataLoader."""
    model.eval()
    all_probs, all_preds, all_targets = [], [], []
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        probs = torch.sigmoid(model(X_batch).squeeze(1))
        preds = (probs >= threshold).long()
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y_batch.numpy())
    return np.array(all_probs), np.array(all_preds), np.array(all_targets)


def evaluate_binary_metrics(model, loader, positive_class_label=1):
    """Compute accuracy, precision, recall, F1, confusion matrix at threshold=0.5."""
    probs, preds, targets = predict_binary(model, loader, threshold=0.5)
    return {
        'accuracy':         accuracy_score(targets, preds),
        'precision':        precision_score(targets, preds, pos_label=positive_class_label, zero_division=0),
        'recall':           recall_score(targets, preds,    pos_label=positive_class_label, zero_division=0),
        'f1':               f1_score(targets, preds, zero_division=0),
        'confusion_matrix': confusion_matrix(targets, preds),
        'probs':            probs,
        'preds':            preds,
        'targets':          targets,
    }

print('Helper functions defined.')

---

## 8. Train the MLP (20 Epochs)

### Training Specification
- **Loss:** `BCEWithLogitsLoss` — combines sigmoid + binary cross-entropy in one numerically stable op
- **Optimizer:** `Adam(lr=1e-3)` — adaptive learning rate, good default for tabular data
- **Epochs:** 20
- **Batch size:** 64

### What to watch
- **Train loss ↓, Val loss ↓** → model is learning
- **Train loss ↓, Val loss ↑** → overfitting (model memorising train set)
- **Both losses plateau** → model has converged

### Accuracy tracking
We compute accuracy each epoch by applying `sigmoid` to logits and thresholding at 0.5.

In [ ]:
model   = SimpleMLP(input_dim=n_features).to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
n_epochs  = 20

train_losses, valid_losses = [], []
train_accs,   valid_accs   = [], []

for epoch in range(1, n_epochs + 1):
    # ── train ──────────────────────────────────────────────────────────────
    tr_loss = train_one_epoch(model, train_loader, optimizer, loss_fn)

    # ── validate ───────────────────────────────────────────────────────────
    va_loss = evaluate_loss(model, valid_loader, loss_fn)

    # ── accuracy (threshold = 0.5) ─────────────────────────────────────────
    _, tr_preds, tr_targets = predict_binary(model, train_loader)
    _, va_preds, va_targets = predict_binary(model, valid_loader)
    tr_acc = accuracy_score(tr_targets, tr_preds)
    va_acc = accuracy_score(va_targets, va_preds)

    train_losses.append(tr_loss);  valid_losses.append(va_loss)
    train_accs.append(tr_acc);     valid_accs.append(va_acc)

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:2d}/{n_epochs} | '
              f'Train loss: {tr_loss:.4f}  acc: {tr_acc:.3f} | '
              f'Val loss: {va_loss:.4f}  acc: {va_acc:.3f}')

print('\nTraining complete.')

---

## 9. Learning Curves and Test Evaluation

### Reading the Learning Curves
- **Left axis (solid lines):** Loss — lower is better
- **Right axis (dashed lines):** Accuracy — higher is better
- A gap between train and val curves suggests overfitting

### Evaluation Metrics for Imbalanced Data

| Metric | Formula | What it measures |
|--------|---------|-----------------|
| Accuracy | (TP+TN)/(TP+TN+FP+FN) | Overall correctness |
| Precision | TP/(TP+FP) | Of predicted bad, how many are truly bad? |
| Recall | TP/(TP+FN) | Of all actual bad, how many did we catch? |
| F1 | 2·P·R/(P+R) | Harmonic mean of precision and recall |

**For credit risk:** Recall (bad-risk class) is often most important — missing a bad borrower
is more costly than falsely flagging a good one.

In [ ]:
# ── LEARNING CURVES ───────────────────────────────────────────────────────
epochs = range(1, n_epochs + 1)
fig, ax1 = plt.subplots(figsize=(8, 4))

ax1.plot(epochs, train_losses, label='Train loss',      color='steelblue')
ax1.plot(epochs, valid_losses, label='Validation loss', color='orange')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')

ax2 = ax1.twinx()
ax2.plot(epochs, train_accs, '--', label='Train accuracy',      color='steelblue')
ax2.plot(epochs, valid_accs, '--', label='Validation accuracy', color='orange')
ax2.set_ylabel('Accuracy'); ax2.set_ylim(0, 1)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
plt.title('Learning Curves — Stratified Split')
plt.tight_layout(); plt.show()

# ── TEST EVALUATION ────────────────────────────────────────────────────────
results = evaluate_binary_metrics(model, test_loader, positive_class_label=1)

print('\n=== Test Results (threshold = 0.5) ===')
print(f'Accuracy : {results["accuracy"]:.4f}')
print(f'Precision (bad-risk class, label=0): {1 - results["precision"]:.4f}')
print(f'Precision (good-risk class, label=1): {results["precision"]:.4f}')
print(f'Recall   (bad-risk class, label=0): {1 - results["recall"]:.4f}')
print(f'Recall   (good-risk class, label=1): {results["recall"]:.4f}')
print(f'F1 Score : {results["f1"]:.4f}')

# Confusion matrix
ConfusionMatrixDisplay(results['confusion_matrix'], display_labels=['bad', 'good']).plot(colorbar=False)
plt.title('Confusion Matrix — Stratified Test Set')
plt.show()

---

## 10. Decision Threshold Analysis

### What is a Decision Threshold?

The model outputs a **probability** (after sigmoid). We convert it to a class label by
comparing against a threshold τ:

```
predicted class = 1 (good)  if  P(good | x) ≥ τ
predicted class = 0 (bad)   if  P(good | x) < τ
```

**Default:** τ = 0.5 (equal cost assumption)

### The Precision-Recall Trade-off

| Lower τ | Higher τ |
|---------|---------|
| More samples predicted as "good" | Fewer samples predicted as "good" |
| Higher recall for "good" class | Higher precision for "good" class |
| More bad-risk borrowers flagged | Fewer bad-risk borrowers flagged |

**For credit risk:** A bank might prefer a **lower threshold** to catch more bad borrowers
(higher recall for bad class), accepting more false alarms.

In [ ]:
def evaluate_at_threshold(model, loader, threshold, positive_class_label=1):
    """Evaluate metrics at a specific decision threshold."""
    _, preds, targets = predict_binary(model, loader, threshold=threshold)
    return {
        'threshold': threshold,
        'accuracy':  accuracy_score(targets, preds),
        'precision': precision_score(targets, preds, pos_label=positive_class_label, zero_division=0),
        'recall':    recall_score(targets, preds,    pos_label=positive_class_label, zero_division=0),
        'f1':        f1_score(targets, preds, zero_division=0),
    }

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

# ── VALIDATION TABLE ───────────────────────────────────────────────────────
print('=== Validation Set — Threshold Comparison ===')
valid_rows = [evaluate_at_threshold(model, valid_loader, t) for t in thresholds]
df_valid = pd.DataFrame(valid_rows).set_index('threshold').round(4)
print(df_valid.to_string())

print('\n=== Test Set — Threshold Comparison ===')
test_rows = [evaluate_at_threshold(model, test_loader, t) for t in thresholds]
df_test = pd.DataFrame(test_rows).set_index('threshold').round(4)
print(df_test.to_string())

# ── PRECISION-RECALL CURVE ─────────────────────────────────────────────────
probs, _, targets = predict_binary(model, valid_loader)
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(targets, probs)

plt.figure(figsize=(7, 5))
plt.plot(recall_curve, precision_curve, color='steelblue', lw=2, label='PR curve')

# Highlight the tested thresholds on the curve
for t in thresholds:
    idx = np.argmin(np.abs(pr_thresholds - t))
    plt.scatter(recall_curve[idx], precision_curve[idx], s=80, zorder=5,
                label=f'τ={t}')

plt.xlabel('Recall (good-risk class)'); plt.ylabel('Precision (good-risk class)')
plt.title('Precision-Recall Curve — Validation Set')
plt.legend(loc='lower left'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('\nKey insight: lowering the threshold increases recall but decreases precision.')

---

## 11. Hyperparameter Tuning with Optuna

### What is Optuna?

Optuna is a **Bayesian hyperparameter optimisation** framework. Instead of exhaustive grid
search, it uses a **TPE (Tree-structured Parzen Estimator)** sampler to intelligently
explore the search space — spending more trials near promising configurations.

### Search Space

| Hyperparameter | Options |
|---------------|---------|
| Hidden layer sizes | [64,32], [128,64], [256,128], [128,32], [256,64], [256,32] |
| Learning rate | 1e-4 to 1e-2 (log scale) |
| Batch normalisation | True / False |
| Optimiser | Adam / AdamW |

### Objective: Maximise Validation Recall (bad-risk class)

We optimise for **recall on the bad-risk class (label=0)** — catching bad borrowers is
the primary business goal.

In [ ]:
class ConfigurableMLP(nn.Module):
    """MLP with optional batch normalisation, configurable hidden sizes."""
    def __init__(self, input_dim, h1=64, h2=32, use_batchnorm=False):
        super().__init__()
        layers = []
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(input_dim))
        layers += [nn.Linear(input_dim, h1)]
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(h1))
        layers.append(nn.ReLU())
        layers += [nn.Linear(h1, h2)]
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(h2))
        layers.append(nn.ReLU())
        layers.append(nn.Linear(h2, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

print('ConfigurableMLP defined.')

In [ ]:
# ── OPTUNA OBJECTIVE ──────────────────────────────────────────────────────
hidden_size_options = [
    (64, 32), (128, 64), (256, 128), (128, 32), (256, 64), (256, 32)
]

def objective(trial):
    # ── Sample hyperparameters ─────────────────────────────────────────────
    h_idx         = trial.suggest_int('hidden_idx', 0, len(hidden_size_options) - 1)
    h1, h2        = hidden_size_options[h_idx]
    lr            = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    use_batchnorm = trial.suggest_categorical('use_batchnorm', [True, False])
    opt_name      = trial.suggest_categorical('optimizer', ['Adam', 'AdamW'])

    # ── Build model ────────────────────────────────────────────────────────
    m = ConfigurableMLP(n_features, h1=h1, h2=h2, use_batchnorm=use_batchnorm).to(device)
    opt_cls = torch.optim.Adam if opt_name == 'Adam' else torch.optim.AdamW
    opt     = opt_cls(m.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()

    # ── Build DataLoader with trial batch size (fixed at 64 here) ─────────
    t_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)
    v_loader = DataLoader(TensorDataset(X_valid_t, y_valid_t), batch_size=64)

    # ── Train 20 epochs ────────────────────────────────────────────────────
    for _ in range(20):
        train_one_epoch(m, t_loader, opt, loss_fn)

    # ── Objective: recall on bad-risk class (label=0) ──────────────────────
    _, preds, targets = predict_binary(m, v_loader, threshold=0.5)
    bad_recall = recall_score(targets, preds, pos_label=0, zero_division=0)
    return bad_recall


# ── RUN STUDY ──────────────────────────────────────────────────────────────
sampler = TPESampler(seed=seed)
study   = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective, n_trials=100)

best_trial  = study.best_trial
best_config = best_trial.params
h1_best, h2_best = hidden_size_options[best_config['hidden_idx']]

print(f'Best trial #{best_trial.number}')
print(f'Best validation recall (bad-risk): {best_trial.value:.4f}')
print('Best hyperparameters:')
for k, v in best_config.items():
    print(f'  {k}: {v}')
print(f'  → hidden sizes: ({h1_best}, {h2_best})')

In [ ]:
# ── RETRAIN BEST MODEL ─────────────────────────────────────────────────────
best_model = ConfigurableMLP(
    n_features,
    h1=h1_best, h2=h2_best,
    use_batchnorm=best_config['use_batchnorm']
).to(device)

opt_cls    = torch.optim.Adam if best_config['optimizer'] == 'Adam' else torch.optim.AdamW
best_opt   = opt_cls(best_model.parameters(), lr=best_config['lr'])
loss_fn    = nn.BCEWithLogitsLoss()

for _ in range(20):
    train_one_epoch(best_model, train_loader, best_opt, loss_fn)

best_results   = evaluate_binary_metrics(best_model, test_loader, positive_class_label=1)
base_results   = results   # from SimpleMLP trained earlier

print('=== Comparison: Baseline SimpleMLP vs Best Optuna Model ===')
print(f'{"Metric":<20} {"Baseline":>10} {"Best Model":>12}')
print('-' * 44)
for metric in ['accuracy', 'precision', 'recall', 'f1']:
    print(f'{metric:<20} {base_results[metric]:>10.4f} {best_results[metric]:>12.4f}')

# ── PRECISION-RECALL CURVES COMPARISON ────────────────────────────────────
probs_base, _, tgt_base = predict_binary(model,      test_loader)
probs_best, _, tgt_best = predict_binary(best_model, test_loader)

p_base, r_base, _ = precision_recall_curve(tgt_base, probs_base)
p_best, r_best, _ = precision_recall_curve(tgt_best, probs_best)

plt.figure(figsize=(7, 5))
plt.plot(r_base, p_base, label='Baseline SimpleMLP', color='steelblue')
plt.plot(r_best, p_best, label='Best Optuna Model',  color='darkorange')
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall Curve — Test Set Comparison')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── HYPERPARAMETER IMPORTANCE ─────────────────────────────────────────────
importance = optuna.importance.get_param_importances(study)

print('\n=== Hyperparameter Importance (descending) ===')
for param, score in sorted(importance.items(), key=lambda x: -x[1]):
    bar = '█' * int(score * 40)
    print(f'  {param:<20} {score:.4f}  {bar}')

# Bar chart
params = list(importance.keys())
scores = [importance[p] for p in params]
plt.figure(figsize=(7, 4))
plt.barh(params, scores, color='steelblue')
plt.xlabel('Importance Score')
plt.title('Hyperparameter Importance — Optuna Study')
plt.tight_layout(); plt.show()

---

## 12. CNN on FashionMNIST

### FashionMNIST
- 70,000 grayscale images (28×28 pixels)
- 10 classes: T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot
- 60,000 train / 10,000 test

### Minimum CNN Architecture

```
Input (1 × 28 × 28)
    ↓
Conv2d(1→32, 3×3) + ReLU + MaxPool2d(2×2)   → (32 × 13 × 13)
    ↓
Conv2d(32→64, 3×3) + ReLU + MaxPool2d(2×2)  → (64 × 5 × 5)
    ↓
Flatten → Linear(64×5×5 → 128) + ReLU
    ↓
Linear(128 → 10)   ← 10-class logits
```

### Why CNNs for images?
- **Parameter sharing:** same filter scans the whole image
- **Local connectivity:** each neuron sees only a small patch
- **Translation invariance:** a cat in the corner = a cat in the centre

In [ ]:
import torchvision
import torchvision.transforms as transforms

# ── LOAD FASHIONMNIST ──────────────────────────────────────────────────────
base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))   # FashionMNIST mean/std
])

train_full = torchvision.datasets.FashionMNIST(root='./data', train=True,  download=True, transform=base_transform)
test_ds    = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=base_transform)

# 80/20 train-validation split from the 60k training set
n_train = int(0.8 * len(train_full))
n_valid = len(train_full) - n_train
train_ds, valid_ds = torch.utils.data.random_split(
    train_full, [n_train, n_valid],
    generator=torch.Generator().manual_seed(seed)
)

train_loader_img = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=0)
valid_loader_img = DataLoader(valid_ds, batch_size=64, shuffle=False, num_workers=0)
test_loader_img  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=0)

class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal',  'Shirt',   'Sneaker',  'Bag',   'Ankle boot']

print(f'Train: {len(train_ds):,}  |  Valid: {len(valid_ds):,}  |  Test: {len(test_ds):,}')

# Visualise a few samples
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img, lbl = train_full[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(class_names[lbl], fontsize=8)
    ax.axis('off')
plt.suptitle('FashionMNIST Samples', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
class SimpleCNN(nn.Module):
    """
    Minimum-spec CNN for FashionMNIST.
    Feature extraction: 2 × (Conv2d → ReLU → MaxPool2d)
    Classifier: 2 fully-connected layers
    """
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # 28×28 → 28×28
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 28×28 → 14×14
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1), # 14×14 → 14×14
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 14×14 → 7×7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

cnn_model = SimpleCNN().to(device)
print(cnn_model)
total_params = sum(p.numel() for p in cnn_model.parameters())
print(f'\nTotal parameters: {total_params:,}')

In [ ]:
# ── CNN TRAINING HELPERS ──────────────────────────────────────────────────
def train_cnn_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total

@torch.no_grad()
def eval_cnn(model, loader, loss_fn):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        total_loss += loss_fn(logits, labels).item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total

# ── TRAIN CNN ─────────────────────────────────────────────────────────────
cnn_loss_fn   = nn.CrossEntropyLoss()
cnn_optimizer = torch.optim.Adam(cnn_model.parameters(), lr=1e-3)
cnn_epochs    = 10

cnn_train_losses, cnn_valid_losses = [], []
cnn_train_accs,   cnn_valid_accs   = [], []

for epoch in range(1, cnn_epochs + 1):
    tr_loss, tr_acc = train_cnn_epoch(cnn_model, train_loader_img, cnn_optimizer, cnn_loss_fn)
    va_loss, va_acc = eval_cnn(cnn_model, valid_loader_img, cnn_loss_fn)

    cnn_train_losses.append(tr_loss); cnn_valid_losses.append(va_loss)
    cnn_train_accs.append(tr_acc);    cnn_valid_accs.append(va_acc)

    print(f'Epoch {epoch:2d}/{cnn_epochs} | '
          f'Train loss: {tr_loss:.4f}  acc: {tr_acc:.3f} | '
          f'Val loss: {va_loss:.4f}  acc: {va_acc:.3f}')

test_loss, test_acc = eval_cnn(cnn_model, test_loader_img, cnn_loss_fn)
print(f'\nTest Accuracy: {test_acc:.4f}')

In [ ]:
# ── CNN LEARNING CURVES ───────────────────────────────────────────────────
epochs_cnn = range(1, cnn_epochs + 1)
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(epochs_cnn, cnn_train_losses, label='Train loss',      color='steelblue')
ax1.plot(epochs_cnn, cnn_valid_losses, label='Validation loss', color='orange')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')

ax2 = ax1.twinx()
ax2.plot(epochs_cnn, cnn_train_accs, '--', label='Train accuracy',      color='steelblue')
ax2.plot(epochs_cnn, cnn_valid_accs, '--', label='Validation accuracy', color='orange')
ax2.set_ylabel('Accuracy'); ax2.set_ylim(0, 1)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
plt.title('CNN Learning Curves — FashionMNIST')
plt.tight_layout(); plt.show()

print(f'Final Test Accuracy: {test_acc:.4f}')

---

## 13. Saliency Maps — CNN Interpretability

### What is a Saliency Map?

A **saliency map** shows which pixels most influence the model's prediction for a given image.

**How it works:**
1. Do a forward pass to get the class score
2. Backpropagate the gradient of that score w.r.t. the **input image** (not the weights)
3. Take the absolute value — large gradient = pixel strongly influences the prediction
4. Visualise as a heatmap overlaid on the original image

```python
image.requires_grad_(True)
score = model(image)[0, target_class]
score.backward()
saliency = image.grad.abs().squeeze()
```

### Why this matters
- Correctly classified: saliency should highlight the object itself
- Misclassified: saliency may focus on background or irrelevant texture
- Helps detect **shortcut learning** (model relying on spurious correlations)

In [ ]:
# ── COLLECT 5 CORRECT + 5 INCORRECT for one target class ─────────────────
TARGET_CLASS = 0   # T-shirt/top  (change to any 0-9)
TARGET_NAME  = class_names[TARGET_CLASS]

cnn_model.eval()

correct_examples   = []   # (image_tensor, true_label, pred_label, confidence)
incorrect_examples = []

with torch.no_grad():
    for imgs, labels in test_loader_img:
        imgs_dev = imgs.to(device)
        logits   = cnn_model(imgs_dev)
        probs    = torch.softmax(logits, dim=1)
        preds    = logits.argmax(1)

        for i in range(len(labels)):
            true_lbl = labels[i].item()
            pred_lbl = preds[i].item()
            conf     = probs[i, pred_lbl].item()

            if true_lbl == TARGET_CLASS and pred_lbl == TARGET_CLASS and len(correct_examples) < 5:
                correct_examples.append((imgs[i], true_lbl, pred_lbl, conf))
            elif true_lbl == TARGET_CLASS and pred_lbl != TARGET_CLASS and len(incorrect_examples) < 5:
                incorrect_examples.append((imgs[i], true_lbl, pred_lbl, conf))

        if len(correct_examples) == 5 and len(incorrect_examples) == 5:
            break

print(f'Target class: {TARGET_NAME}')
print(f'Correct examples found:   {len(correct_examples)}')
print(f'Incorrect examples found: {len(incorrect_examples)}')

In [ ]:
def compute_saliency(model, image_tensor, target_class):
    """
    Compute vanilla gradient saliency map.
    Returns saliency as a 2D numpy array (H x W), normalised to [0, 1].
    """
    model.eval()
    img = image_tensor.unsqueeze(0).to(device)   # (1, 1, 28, 28)
    img.requires_grad_(True)

    logits = model(img)
    score  = logits[0, target_class]
    model.zero_grad()
    score.backward()

    # Take max over colour channels (only 1 here), then normalise
    saliency = img.grad.data.abs().squeeze().cpu().numpy()  # (28, 28)
    saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)
    return saliency


def unnorm(tensor):
    """Reverse FashionMNIST normalisation for display."""
    return tensor.squeeze().cpu().numpy() * 0.3530 + 0.2860


# ── VISUALISE 10 EXAMPLES (5 correct + 5 incorrect) ──────────────────────
all_examples = correct_examples + incorrect_examples
labels_row   = ['CORRECT'] * 5 + ['INCORRECT'] * 5

fig, axes = plt.subplots(10, 2, figsize=(6, 22))
fig.suptitle(f'Saliency Maps — Target Class: {TARGET_NAME}', fontsize=13, y=1.01)

for row, ((img_t, true_lbl, pred_lbl, conf), status) in enumerate(zip(all_examples, labels_row)):
    sal = compute_saliency(cnn_model, img_t, pred_lbl)
    img_np = unnorm(img_t)

    # Original image
    axes[row, 0].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    axes[row, 0].set_title(
        f'{status}\nTrue: {class_names[true_lbl]}  Pred: {class_names[pred_lbl]}\nConf: {conf:.2%}',
        fontsize=8, color='green' if status == 'CORRECT' else 'red'
    )
    axes[row, 0].axis('off')

    # Saliency map
    axes[row, 1].imshow(sal, cmap='hot')
    axes[row, 1].set_title('Saliency Map', fontsize=8)
    axes[row, 1].axis('off')

plt.tight_layout(); plt.show()

### Interpreting Saliency Maps

**What to look for:**

| Example type | Healthy saliency | Concerning saliency |
|-------------|-----------------|-------------------|
| Correct | Highlights the garment shape/texture | Focuses on background |
| Incorrect | Highlights features of the *predicted* class | Ignores the actual object |

**Common observations on FashionMNIST:**
- Correctly classified T-shirts: saliency concentrates on the collar/sleeve region
- Misclassified T-shirts (predicted as Shirt): saliency may focus on similar texture patterns
- Background pixels with high saliency → model may be using spurious correlations

**Limitations of vanilla saliency:**
- Can be noisy — consider SmoothGrad or Integrated Gradients for cleaner maps
- Shows *where* the model looks, not *why* it makes a decision
- See [I14 - Explainable AI](../Intermediate/I14%20-%20Explainable%20AI%20and%20Interpretability.ipynb) for SHAP and LIME alternatives

---

## 14. Key Takeaways

### Part 1 — Tabular MLP

| Concept | Key point |
|---------|-----------|
| Stratified split | Preserves class ratio — essential for imbalanced data |
| Preprocessing | Fit on train only — prevents data leakage |
| BCEWithLogitsLoss | Numerically stable binary cross-entropy (sigmoid built-in) |
| 1 output neuron | Sufficient for binary classification |
| Decision threshold | Default 0.5 assumes equal cost; lower τ → higher recall |
| Optuna TPE | Smarter than grid search — focuses on promising regions |

### Part 2 — CNN + Saliency

| Concept | Key point |
|---------|-----------|
| Conv2d → ReLU → MaxPool | Standard feature extraction block |
| CrossEntropyLoss | Multi-class loss (softmax built-in) |
| Saliency map | Gradient of class score w.r.t. input pixels |
| Correct vs incorrect | Saliency reveals what the model focuses on |

### Next Steps
- [I02 - Regularization Techniques](../Intermediate/I02%20-%20Regularization%20Techniques.ipynb) — Dropout, weight decay
- [I03 - Batch and Layer Normalization](../Intermediate/I03%20-%20Batch%20and%20Layer%20Normalization.ipynb) — BatchNorm in depth
- [I04 - Advanced CNN Architectures](../Intermediate/I04%20-%20Advanced%20CNN%20Architectures.ipynb) — ResNet, VGG
- [I10 - Hyperparameter Tuning](../Intermediate/I10%20-%20Hyperparameter%20Tuning%20and%20AutoML.ipynb) — Advanced Optuna patterns
- [I14 - Explainable AI](../Intermediate/I14%20-%20Explainable%20AI%20and%20Interpretability.ipynb) — SHAP, LIME, GradCAM

---

## Learning Progress Tracker

### Completion Status
- [ ] Lesson completed
- [ ] All code cells executed successfully
- [ ] Understood all key concepts

### Understanding Level
Rate your understanding (1-5): _____ / 5

### Notes & Reflections
```
- What was challenging?
- What surprised you?
- How does this connect to your coursework?
```

---